# M0 · GPU 就绪与实验纪律

**学习目标：** 在提出任何优化结论之前，先证明：(1) 你测量的 GPU 确实是你以为的那块 GPU；(2) 你的计时方式尊重 GPU 异步性；(3) 你报告的每个数字都有明确的来源声明。

参考讲义：[M00 讲义](../course/M00_setup_and_discipline/LESSON.zh-CN.md) · 切换语言：[English](./00_experimental_discipline.ipynb)

## 三种课程模式

| 模式 | 运行内容 | 数据位置 | 适用场景 |
|---|---|---|---|
| `REFERENCE` | 仅分析已冻结的历史证据（CPU、离线） | `evidence/` | 学习方法论、CPU CI、调试 |
| `GPU_CLASSROOM` | **你自己在这台 W7900 上的有界实测** | `runs/` | Radeon Cloud 课堂默认模式 |
| `GPU_RESEARCH` | 完整协议的物理 A/B 实验 | `runs/` | 毕业课题 / 研究级复现 |

在第一个代码单元中设置 `COURSE_MODE = "GPU_CLASSROOM"`，然后
**Kernel → Restart & Run All**。课堂练习是一次*有界物理实验*：它证明流水线
在你的 GPU 上真实可用，并给出初步对比——但它**不是**正式性能结论（正式结论
需要研究协议：多提示词 × 多种子重复、匹配的样本总体）。
`evidence/` 中的历史数字是用来研究对照的参考值，永远不能替代你自己的测量。

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="00")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

## 1 · 硬件准入 —— 这真的是一块 W7900 吗？
GPU 模式下，上方的准入横幅已经给出答案。下面的单元记录物理设备事实，以及本课程锁定的模型文件（体积 + 锁定的 sha256 摘要——信任锚点在 `lab/weights.py`，绝不依赖在线端点）。

In [ ]:
from lab import weights as weights_mod

if session.mode != "REFERENCE":
    d = session.admission.detail
    _stamp(MEASURED_TODAY,
           f"device={d['device_name']} gfx={d['gfx_arch']} "
           f"vram={d['vram_used_mib']}/{d['vram_total_mib']} MiB hip={d['hip']}")
else:
    print("REFERENCE mode: physical admission is skipped by design; "
          "the frozen campaign device facts are in evidence/results/env/")
for state in weights_mod.weights_state():
    r = weights_mod.verify_weight(state["rel"])
    print(f"  {state['state']:8s} {state['rel'].split('/')[-1]}  ({r['reason']})")

## 2 · GPU 异步性 —— 为什么不同步的计时是错的
GPU 会在工作完成之前就交还控制权。启动内核后立刻读计时器，测到的是*启动*而不是*执行*。下面的单元在 GPU 模式下用一次大矩阵乘法物理演示这一点；REFERENCE 模式下用 CPU 线程类比，保证这一课永远不会被跳过。

In [ ]:
import time

def demo_async_gpu():
    import torch
    torch.manual_seed(0)
    a = torch.randn(4096, 4096, device="cuda", dtype=torch.bfloat16)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(20):
        b = a @ a
    t_no_sync = time.perf_counter() - t0      # WRONG: measures launch queue
    torch.cuda.synchronize()                  # correct: wait for completion
    t_synced = time.perf_counter() - t0       # RIGHT: includes execution
    print(f"WITHOUT sync: {t_no_sync:.4f}s   WITH sync: {t_synced:.4f}s")
    assert t_synced >= t_no_sync
    _stamp(MEASURED_TODAY, f"sync gap = {t_synced - t_no_sync:.3f}s on this GPU")

def demo_async_cpu_threads():
    import threading
    done = threading.Event()
    result = {}
    def work():
        x = sum(i * i for i in range(3_000_000))
        result["x"] = x
        done.set()
    t0 = time.perf_counter()
    threading.Thread(target=work).start()
    t_no_join = time.perf_counter() - t0      # WRONG: worker still running
    done.wait()                               # correct: join first
    t_joined = time.perf_counter() - t0
    print(f"WITHOUT join: {t_no_join:.5f}s   WITH join: {t_joined:.5f}s")
    _stamp(ESTIMATE, "CPU-thread analogy of GPU asynchrony (not a GPU measurement)")

gpu_cell(demo_async_gpu) or demo_async_cpu_threads()

## 3 · Smoke / cold / warm —— 三种不同的样本总体
**smoke** 运行证明流水线可以执行（单次、无预热、无统计）。**cold** 运行包含首次加载成本（权重读盘、分配器增长）。**warm** 运行才是延迟结论可比的总体。把它们混在一起是经典的测量错误。在冻结实验数据中找每种真实样本：

In [ ]:
from lab import notebook_support as ns
import statistics
import json

kinds = {}
for d in ns.read_json("lab/reference_runs.json")["reference_run_dirs"]["B0"]:
    for f in sorted((ROOT / "evidence/results" / d).glob("*.json")):
        rec = json.load(open(f))
        kinds.setdefault(rec["run_kind"], 0)
        kinds[rec["run_kind"]] += len(rec.get("runs", []))
print("B0 reference populations:", kinds)
_stamp(HISTORICAL, "population labels above come from the frozen campaign")

## 4 · 动手练习
1. 预测你这台实例上 B0 的 warm/cold 差距，然后在 GPU_CLASSROOM 模式下与 M1 对比。
2. 在冻结记录里找出某个 warm 样本比另一会话 cold 样本更慢的例子——为什么“cold 一定最慢”不是不变量？
3. 解释为什么 smoke PASS 永远不能证明加速比。

## 验收标准

- [ ] 你能指出哪些数字是**今天在这块 GPU 上实测的**、哪些是
      **历史参考值**、哪些是**估计/模拟**。
- [ ] 你产生的每次物理运行都在 `runs/` 下有可见的 `run_id`。
- [ ] 你没有把 smoke/课堂样本冒充为正式协议结果。
- [ ] 你能解释你报告的每项测量的一个局限性。